In [1]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
from pprint import pprint

from joblib import Parallel, delayed
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
if torch.backends.mps.is_available():
    device = torch.device('mps')
    device = torch.device('cpu')
elif torch.cuda.is_available():
    device = torch.device('cuda')
print(f'torch device: {device}')

# GNN package
from torch_geometric.loader import DataLoader

import networkx as nx

torch device: cuda


/home/alan/miniconda3/envs/seg_match/lib/python3.9/site-packages/torch_geometric/typing.py:31: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/alan/miniconda3/envs/seg_match/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so: undefined symbol: _ZN3c106SymInt19promote_to_negativeEv
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "


In [2]:
from dataset import CustomDataset, transform_data
from models import CustomModel
from losses import CustomCriterion
from eva import compute_eval

torch device: cuda


In [4]:
SOLO_NAME =  'poisson3r8_vis'
SCENE = 'SimpleOffice'
# SCENE = 'WP16'
DATA_DIR = f'data/{SCENE}/{SOLO_NAME}'

In [5]:
ds = CustomDataset(root=DATA_DIR, scene=SCENE, transform=transform_data)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
dc = next(iter(dl))

In [6]:
dc.keys()

dict_keys(['bbox_embs', 'g1_rgb_emb', 'g2_rgb_emb', 'e1i', 'e1j', 'e2i', 'e2j', 'e1i_count', 'e1j_count', 'e2i_count', 'e2j_count', 'total_obj_count', 'total_node_count', 'g1_node_count', 'g2_node_count', 'g1_edge_count', 'g2_edge_count', 'g1_camera_pose', 'g2_camera_pose', 'g1_camera_intrinsics', 'g2_camera_intrinsics', 'g1_step', 'g2_step', 'obj_pose_in_W', 'bbox', 'node_attr', 'edge_index', 'edge_attr'])

In [7]:
# ds = CustomDataset(root=DATA_DIR, scene=SCENE)
# ds[0]['node_df'].columns

In [8]:
# # e1is = []
# len_e2 = []
# for dc in iter(dl):
#   e1is.append(len(dc['e1i'][0]))
#   len_e2.append(len(dc['e2i'][0]) + len(dc['e2j'][0]))

# len_e2 = np.array(len_e2)
# for k in [1, 3, 5]:
#   p_rand = np.clip(1 - (len_e2 - k) / len_e2, 0, 1)
#   print(f'rand hits@{k}: {p_rand.mean():.4f}, {p_rand.std():.4f}')

In [35]:
node_attr_dim = dc['node_attr'].shape[2]
node_visual_dim = dc['bbox_embs'].shape[2]
edge_attr_dim = dc['edge_attr'].shape[2]
emb_dim = 512

print(f'node attr dim: {node_attr_dim}')
print(f'node visual dim: {node_visual_dim}')
print(f'edge attr dim: {edge_attr_dim}')
print(f'emb dim: {emb_dim}')

node attr dim: 31
node visual dim: 1000
edge attr dim: 3
emb dim: 512


In [36]:
ds = CustomDataset(root=DATA_DIR, scene=SCENE, transform=transform_data)
# random select 1000 samples from ds
# ds = torch.utils.data.Subset(ds, np.random.choice(len(ds), 1000, replace=False))
train_ds, eval_ds, test_ds = torch.utils.data.random_split(ds, [0.5, 0.2, 0.3])
# # # train_ds = torch.utils.data.Subset(train_ds, [0])  # for debugging

# # # Do not use batch_size>1 for now, it will crash the e1i, e1j, e2i, e2j indices
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)
eval_dl = DataLoader(eval_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

print('train size:', len(train_ds))
print('eval size:', len(eval_ds))
print('test size:', len(test_ds))

train size: 562
eval size: 224
test size: 336


In [37]:
def train_step(model, data_dict, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    pred_dict = model(data_dict)
    loss = criterion(pred_dict, data_dict)
    loss.backward()
    optimizer.step()
    return loss.item()

In [38]:
def eval_step(model, data_dict, criterion):
    model.eval()
    # with torch.no_grad():
    pred_dict = model(data_dict)
    loss = criterion(pred_dict, data_dict)
    metrics = compute_eval(pred_dict, data_dict)
    return loss.item(), metrics

In [39]:
model = CustomModel(node_attr_dim, node_visual_dim, edge_attr_dim, emb_dim, match_threshold=0.2).to(device)
criterion = CustomCriterion(device)
optimizer = optim.Adam(model.parameters(), lr=5e-4)

e = 0
history = {'train_loss': [], 'eval_loss': [], 'eval_metrics': []}

In [40]:
def data_dict_to_device(data_dict, device):
    for k in data_dict:
        if isinstance(data_dict[k], torch.Tensor):
            data_dict[k] = data_dict[k].to(device)
    return data_dict

In [41]:
# training loop
for _ in tqdm(range(200)):
    e += 1
    train_loss = 0
    for data_dict in train_dl:
        data_dict = data_dict_to_device(data_dict, device)
        train_loss += train_step(model, data_dict, optimizer, criterion)
    train_loss /= len(train_dl)

    if e % 10 == 0:
        mean_metrics = {}
        sum_eval_loss = 0

        with torch.no_grad():
            model.eval()
            for data_dict in eval_dl:
                data_dict = data_dict_to_device(data_dict, device)
                eval_loss, metrics = eval_step(model, data_dict, criterion)
                sum_eval_loss += eval_loss
                for k, v in metrics.items():
                    if k in mean_metrics:
                        mean_metrics[k] = mean_metrics.get(k, 0) + v 
                    else:
                        mean_metrics[k] = v

        for k, v in mean_metrics.items():
            mean_metrics[k] = v / len(eval_dl)
        eval_loss = sum_eval_loss / len(eval_dl)

        history['train_loss'].append(train_loss)
        history['eval_loss'].append(eval_loss)
        history['eval_metrics'].append(mean_metrics)
        print(f'----- epoch: {e} -----')
        print(f'train loss: {train_loss:.6f}')
        print(f'eval  loss: {eval_loss:.6f}')
        print('eval metrics:')
        for k, v in mean_metrics.items():
            print(f'{k}: {v:.4f}')
        print()

plt.plot(history['train_loss'], label='train loss')
plt.plot(history['eval_loss'], label='eval loss')
plt.legend()
plt.show()

  0%|          | 0/200 [00:00<?, ?it/s]

----- epoch: 10 -----
train loss: 1.389892
eval  loss: 1.336062
eval metrics:
acc: 0.5471
pre: 0.0000
rec: 0.0000
f1: 0.0000

----- epoch: 20 -----
train loss: 1.389892
eval  loss: 1.336062
eval metrics:
acc: 0.5471
pre: 0.0000
rec: 0.0000
f1: 0.0000



In [1]:
mean_metrics = {}
sum_test_loss = 0
gid2pred = {}

with torch.no_grad():
    model.eval()
    for data_dict in tqdm(test_dl):
        data_dict = data_dict_to_device(data_dict, device)
        test_loss, metrics = eval_step(model, data_dict, criterion)
        sum_test_loss += test_loss

        pred_dict = model(data_dict) 

        g1_step = data_dict['g1_step'][0].item()
        if g1_step not in gid2pred:
            gid2pred[g1_step] = {'pred': [pred_dict], 'data': [data_dict]}
        else:
            gid2pred[g1_step]['pred'].append(pred_dict)
            gid2pred[g1_step]['data'].append(data_dict)

        for k, v in metrics.items():
            if k in mean_metrics:
                mean_metrics[k] = mean_metrics[k] + v
            else:
                mean_metrics[k] = v
        


for k, v in mean_metrics.items():
    mean_metrics[k] = v / len(test_dl)

mean_test_loss = sum_test_loss / len(test_dl)
print(f'mean_test loss: {mean_test_loss:.4f}')
print('test metrics:')
for k, v in mean_metrics.items():
    print(f'{k}: {v:.4f}')

NameError: name 'torch' is not defined

In [41]:
res = []

for gid, v in gid2pred.items():
    g1_pose = v['data'][0]['g1_camera_pose'][0]
    g1_step = v['data'][0]['g1_step'][0].item()

    pred = []
    gt_pose = None
    gt_step = -1
    gt_pose_loss = np.inf

    for idx in range(len(v['pred'])):
        pred_dict = v['pred'][idx]
        data_dict = v['data'][idx]

        g1_pose = data_dict['g1_camera_pose'][0]
        g1_step = data_dict['g1_step'][0].item()
        g2_pose = data_dict['g2_camera_pose'][0]
        g2_step = data_dict['g2_step'][0].item()

        # pred
        match_mask = (pred_dict['matches0'] != -1)
        if sum(match_mask) == 0:
            score = -np.inf
            continue
        else:
            match_score = pred_dict['matching_scores0'][match_mask].sort(descending=True)[0][:3]
            score = torch.mean(match_score).item()
        pred.append((g2_step, score, g2_pose.numpy()))

        # gt
        loss = np.linalg.norm(g1_pose - g2_pose).item()
        if loss < gt_pose_loss:
            gt_pose_loss = loss
            gt_pose = g2_pose.numpy()
            gt_step = g2_step

    pred = sorted(pred, key=lambda x: x[1], reverse=True)


    # print(f'g1_step: {g1_step}')
    # print(f'gt_step: {gt_step}')
    # print(f'pred: {[p[0] for p in pred]}')
    # print()
    res.append({
        'g1_step': g1_step, 
        'gt_step': gt_step,
        'gt_pose': gt_pose,
        'pred': pred,
    })

res = sorted(res, key=lambda x: x['g1_step'])
n_data = len(res)
n_no_match = sum([len(d['pred']) == 0 for d in res])
n_hits_at_1 = sum([True for d in res if len(d['pred']) > 0 and np.allclose(d['pred'][0][2], d['gt_pose'])])
print(f'data len: {n_data}')
print(f'num of no matches: {n_no_match}')
# print(f'num of matches: {sum([len(d["pred"]) != 0 for d in res])}')
print(f'hits@1: {n_hits_at_1} ({n_hits_at_1 / (n_data - n_no_match) * 100:.4f}%)')


data len: 116
num of no matches: 14
hits@1: 57 (55.8824%)


In [ ]:
data_dict = list(iter(test_dl))[10]
pred_dict = model(data_dict)
cam_pose = data_dict['g1_camera_pose']

with torch.no_grad():
    metrics = compute_eval(pred_dict, data_dict)
    for k, v in metrics.items():
        if type(v) == torch.Tensor:
            metrics[k] = v.item()
pprint(metrics)

{'acc': 0.5757575757575758,
 'f1': 0.0,
 'hits@1': 0.6,
 'hits@3': 1.0,
 'hits@5': 1.0,
 'pre': 0.0,
 'rec': 0.0}


In [ ]:
# turn e1i and e1j into numpy array
e1i = data_dict['e1i'].squeeze(0).numpy()
e1j = data_dict['e1j'].squeeze(0).numpy()
e2i = data_dict['e2i'].squeeze(0).numpy() - data_dict['g1_node_count'].item()
e2j = data_dict['e2j'].squeeze(0).numpy() - data_dict['g2_node_count'].item()
e1 = np.concatenate([e1i, e1j])
e2 = np.concatenate([e2i, e2j])
print('g1 node count', data_dict['g1_node_count'].item())
print('g2 node count', data_dict['g2_node_count'].item())
print(f'e1i: {e1i.shape}, e2i: {e2i.shape}, e1j: {e1j.shape}, e2j: {e2j.shape}')
# print(f'e1: {e1}')
# print(f'e2: {e2}')

g1 node count 19
g2 node count 15
e1i: (4,), e2i: (4,), e1j: (15,), e2j: (11,)


In [ ]:
gt_match = [[i, j] for i, j in zip(e1i, e2i)]
pred_match = [[i, j.item()] for i, j in enumerate(pred_dict['matches0']) if j != -1]

print(f'gt match:')
pprint(gt_match)
print(f'pred match:')
pprint(pred_match)

gt match:
[[1, 2], [7, 12], [8, 14], [6, 9]]
pred match:
[[1, 2], [2, 0], [5, 4], [18, 14]]


In [ ]:
# gt_match = [[i, j] for i, j in zip(e1i, e2i)]
# print(f'gt match:')
# pprint(gt_match)

# g1_node_count = data_dict['g1_node_count'][0].detach().cpu().numpy()
# emb = pred_dict['joint_embs'].squeeze(0).detach().cpu().numpy()
# emb = emb / np.linalg.norm(emb, axis=1)[:, None]
# emb1 = emb[:data_dict['g1_node_count']]
# emb2 = emb[data_dict['g1_node_count']:]
# W = emb1 @ emb2.T
# corr_ids = np.argmax(W, axis=1) + g1_node_count

# G = nx.Graph()
# for i in range(W.shape[0]):
#     for j in range(W.shape[1]):
#         G.add_edge(i, j + g1_node_count, weight=W[i, j])

# match = nx.bipartite.minimum_weight_full_matching(G, corr_ids)
# match = [[i, j] for i, j in match.items() if i < data_dict['g1_node_count'].item()]
# print(f'pred match:')
# pprint(match)

# match_G = nx.Graph()
# for i in match.keys():
#     j = match[i]
#     match_G.add_edge(i, j, weight=G[i][j]['weight'])
# pos = nx.bipartite_layout(match_G, match.keys())
# nx.draw(match_G, pos, node_color='lightblue', with_labels=True, node_size=500)